In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# NOTEBOOK B1 — ViT-B/16 — PCam — SimCLR — 1 split × 3 seeds
# CRITICAL FIX: train-fitted feature standardizer applied to val.
# Checkpoint: /kaggle/working/_partial_results_raw.csv
# ============================================================

import os, time, random
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
import h5py

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
USE_AMP = (DEVICE.type == "cuda")

# Protocol (ViT)
N_SPLITS = 1
SEEDS = [0, 1, 2]
SPLIT_BASE_SEED = 1000
LABEL_FRACS = [0.01, 0.05, 0.10]
VAL_RATIO = 0.2

SSL_EPOCHS = 10
PROBE_EPOCHS = 10
LR_SSL = 1e-3
LR_PROBE = 1e-4

SAMPLE_FRAC_PCAM = 0.2

BATCH_SSL_VIT = 16
BATCH_SUP_VIT = 32
IMG_PCAM_VIT = 224

NUM_WORKERS_PATH = 4
NUM_WORKERS_H5 = 0
PIN_MEMORY = True

PARTIAL_RAW_PATH = "/kaggle/working/_partial_results_raw.csv"

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_any_under_input(pattern: str) -> List[Path]:
    return list(Path("/kaggle/input").rglob(pattern))

def pick_shortest(paths: List[Path]) -> Optional[Path]:
    return sorted(paths, key=lambda p: len(str(p)))[0] if paths else None

def sanitize_np_probs(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs).reshape(-1)
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    return np.clip(probs, 1e-7, 1 - 1e-7)

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (probs >= lo) & (probs < hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(labels[m].mean() - probs[m].mean())
    return float(ece)

def compute_metrics_binary(probs: np.ndarray, labels: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    preds = (probs >= thr).astype(int)
    acc = float((preds == labels).mean())
    try:
        auc = float(roc_auc_score(labels, probs))
    except ValueError:
        auc = float("nan")
    f1 = float(f1_score(labels, preds, zero_division=0))
    brier = float(brier_score_loss(labels, probs))
    ece = compute_ece(probs, labels)
    return {"acc": acc, "auc": auc, "f1": f1, "ece": ece, "brier": brier}

def sanitize_tensor(x: torch.Tensor, clamp: float = 1e4) -> torch.Tensor:
    x = x.float()
    x = torch.nan_to_num(x, nan=0.0, posinf=clamp, neginf=-clamp)
    return torch.clamp(x, -clamp, clamp)

@torch.no_grad()
def fit_standardizer(train_feats: torch.Tensor, eps: float = 1e-6):
    mu = train_feats.float().mean(dim=0, keepdim=True)
    sd = train_feats.float().std(dim=0, keepdim=True).clamp_min(eps)
    return mu, sd

@torch.no_grad()
def apply_standardizer(feats: torch.Tensor, mu: torch.Tensor, sd: torch.Tensor):
    return (feats.float() - mu) / sd

class TwoViewSSL_Path(Dataset):
    def __init__(self, image_paths: List[str], img_size: int):
        self.image_paths = image_paths
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), self.t(img)

class Supervised_Path(Dataset):
    def __init__(self, image_paths: List[str], labels: List[int], img_size: int):
        self.image_paths, self.labels = image_paths, labels
        self.t = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), int(self.labels[idx])

class DataSourceBase:
    def __init__(self, name): self.name = name
    def __len__(self): raise NotImplementedError
    def labels_all(self): raise NotImplementedError
    def make_ssl_dataset(self, indices, img_size): raise NotImplementedError
    def make_sup_dataset(self, indices, img_size): raise NotImplementedError

class PathDataSource(DataSourceBase):
    def __init__(self, name, paths, labels):
        super().__init__(name)
        self.paths = list(paths)
        self.labels = list(map(int, labels))
    def __len__(self): return len(self.paths)
    def labels_all(self): return np.asarray(self.labels, dtype=np.int64)
    def make_ssl_dataset(self, indices, img_size):
        return TwoViewSSL_Path([self.paths[i] for i in indices], img_size)
    def make_sup_dataset(self, indices, img_size):
        p = [self.paths[i] for i in indices]
        y = [self.labels[i] for i in indices]
        return Supervised_Path(p, y, img_size)

def load_pcam_datasource(sample_frac: float = 1.0) -> DataSourceBase:
    train_labels_paths = find_any_under_input("train_labels.csv")
    if not train_labels_paths:
        raise FileNotFoundError("PCam train_labels.csv not found under /kaggle/input. Add histopathologic-cancer-detection.")
    labels_path = str(pick_shortest(train_labels_paths))
    pcam_root = str(Path(labels_path).parent)
    df = pd.read_csv(labels_path)
    if 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)

    def make_path(img_id: str):
        tif = os.path.join(pcam_root, "train", f"{img_id}.tif")
        if os.path.exists(tif):
            return tif
        for ext in ["png","jpg","jpeg"]:
            p = os.path.join(pcam_root, "train", f"{img_id}.{ext}")
            if os.path.exists(p): return p
        return tif

    df["image_path"] = df["id"].astype(str).apply(make_path)
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    ds = PathDataSource("pcam", df["image_path"].tolist(), df["label"].astype(int).tolist())
    ds._universe_indices = np.arange(len(ds), dtype=np.int64)
    return ds

def make_splits_indices(universe_indices, labels, n_splits, val_ratio, base_seed):
    labels = np.asarray(labels, dtype=np.int64)
    splits = []
    for split_id in range(n_splits):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=base_seed + split_id)
        tr_rel, va_rel = next(sss.split(np.zeros(len(universe_indices)), labels[universe_indices]))
        splits.append((universe_indices[tr_rel], universe_indices[va_rel]))
    return splits

def stratified_label_subset_indices(train_indices, labels, frac, seed):
    labels = np.asarray(labels, dtype=np.int64)
    n = len(train_indices)
    k = max(1, int(n * frac))
    sss = StratifiedShuffleSplit(n_splits=1, train_size=k, random_state=seed)
    sel_rel, _ = next(sss.split(np.zeros(n), labels[train_indices]))
    return train_indices[sel_rel]

def get_vit_b_16(pretrained: bool = False):
    try:
        from torchvision.models import ViT_B_16_Weights
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        m = models.vit_b_16(weights=weights)
    except Exception:
        m = models.vit_b_16(pretrained=pretrained)
    try:
        m.heads = nn.Identity()
    except Exception:
        try: m.heads.head = nn.Identity()
        except Exception: pass
    feat_dim = getattr(m, "hidden_dim", 768)
    return m, feat_dim

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim=128, hidden_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )
    def forward(self, x): return self.net(x)

def get_simclr_encoder_vit():
    backbone, feat_dim = get_vit_b_16(pretrained=False)
    proj = ProjectionHead(feat_dim, proj_dim=128, hidden_dim=1024)
    return nn.Sequential(backbone, proj)

def nt_xent_loss_fp32(z1, z2, temperature=0.5):
    z1 = z1.float(); z2 = z2.float()
    b = z1.size(0)
    z1 = F.normalize(z1, dim=1); z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / float(temperature)
    sim = sim.masked_fill(torch.eye(2*b, dtype=torch.bool, device=z.device), -1e4)
    labels = torch.arange(b, device=z.device)
    labels = torch.cat([labels + b, labels], dim=0)
    return F.cross_entropy(sim, labels)

def pretrain_simclr_oomsafe(ssl_ds: Dataset, batch_ssl: int, epochs: int, lr: float, num_workers: int):
    try_bss = [batch_ssl, 12, 8, 6, 4, 2]
    last_err = None
    for bs in try_bss:
        try:
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
            enc = get_simclr_encoder_vit().to(DEVICE)
            opt = torch.optim.Adam(enc.parameters(), lr=lr)
            dl = DataLoader(ssl_ds, batch_size=bs, shuffle=True, num_workers=num_workers,
                            pin_memory=PIN_MEMORY, drop_last=True)
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
            enc.train()
            for ep in range(1, epochs+1):
                t0 = time.time()
                total, n = 0.0, 0
                for x1, x2 in dl:
                    x1 = x1.to(DEVICE, non_blocking=True)
                    x2 = x2.to(DEVICE, non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    with torch.amp.autocast("cuda", enabled=USE_AMP):
                        z1 = enc(x1); z2 = enc(x2)
                    loss = nt_xent_loss_fp32(z1, z2)
                    if not torch.isfinite(loss):
                        continue
                    scaler.scale(loss).backward()
                    scaler.step(opt); scaler.update()
                    total += float(loss.item()) * x1.size(0)
                    n += x1.size(0)
                print(f"[SimCLR/VIT] ep {ep}/{epochs} loss={total/max(1,n):.4f} time={time.time()-t0:.1f}s (bs={bs})")
            return enc, bs
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and DEVICE.type == "cuda":
                print(f"[OOM] bs={bs} failed, reducing...")
                last_err = e
                continue
            raise
    raise last_err

@torch.no_grad()
def extract_features(backbone: nn.Module, dl: DataLoader):
    backbone.eval()
    backbone = backbone.to(DEVICE)
    feats, ys = [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            z = backbone(x)
        if isinstance(z, (tuple, list)): z = z[0]
        if z.ndim > 2: z = torch.flatten(z, 1)
        feats.append(sanitize_tensor(z).detach().cpu())
        ys.append(y.numpy())
    return sanitize_tensor(torch.cat(feats, 0)), np.concatenate(ys, 0).astype(int)

class LinearProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, 1)
    def forward(self, x): return self.fc(x).squeeze(1)

def train_probe(feats_std: torch.Tensor, y: np.ndarray, epochs: int, lr: float):
    X = sanitize_tensor(feats_std).float()
    y_t = torch.tensor(y, dtype=torch.float32)
    dl = DataLoader(torch.utils.data.TensorDataset(X, y_t), batch_size=256, shuffle=True)
    clf = LinearProbe(X.size(1)).to(DEVICE)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss()
    clf.train()
    for ep in range(1, epochs+1):
        total, n = 0.0, 0
        for xb, yb in dl:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            loss = crit(sanitize_tensor(clf(xb)), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            opt.step()
            total += float(loss.item()) * xb.size(0)
            n += xb.size(0)
        print(f"[Probe] ep {ep}/{epochs} loss={total/max(1,n):.4f}")
    return clf

@torch.no_grad()
def eval_probe(clf: nn.Module, feats_std: torch.Tensor, y: np.ndarray):
    clf.eval()
    X = sanitize_tensor(feats_std).float().to(DEVICE)
    logits = sanitize_tensor(clf(X).detach().cpu())
    probs = sanitize_np_probs(torch.sigmoid(logits).numpy())
    return compute_metrics_binary(probs, y)

def append_checkpoint(rows):
    if not rows: return
    os.makedirs("/kaggle/working", exist_ok=True)
    pd.DataFrame(rows).to_csv(PARTIAL_RAW_PATH, index=False)

# -----------------------
# MAIN
# -----------------------
os.makedirs("/kaggle/working", exist_ok=True)
if os.path.exists(PARTIAL_RAW_PATH):
    os.remove(PARTIAL_RAW_PATH)

ds = load_pcam_datasource(sample_frac=SAMPLE_FRAC_PCAM)
uni = getattr(ds, "_universe_indices", np.arange(len(ds), dtype=np.int64))
labels_all = ds.labels_all()

splits = make_splits_indices(uni, labels_all, N_SPLITS, VAL_RATIO, SPLIT_BASE_SEED)
use_workers = NUM_WORKERS_PATH

rows = []
for split_id, (tr_idx, va_idx) in enumerate(splits):
    for seed in SEEDS:
        set_seed(10_000 + 100*split_id + seed)
        print("\n" + "="*90)
        print(f"[RUN] PCam ViT SimCLR split={split_id} seed={seed} n_tr={len(tr_idx)} n_va={len(va_idx)}")
        print("="*90)

        ssl_ds = ds.make_ssl_dataset(tr_idx, img_size=IMG_PCAM_VIT)
        enc, used_bs = pretrain_simclr_oomsafe(ssl_ds, BATCH_SSL_VIT, SSL_EPOCHS, LR_SSL, use_workers)
        backbone = enc[0]  # ViT backbone

        val_ds = ds.make_sup_dataset(va_idx, img_size=IMG_PCAM_VIT)
        val_dl = DataLoader(val_ds, batch_size=BATCH_SUP_VIT, shuffle=False, num_workers=use_workers, pin_memory=PIN_MEMORY)

        for frac in LABEL_FRACS:
            sub_tr_idx = stratified_label_subset_indices(tr_idx, labels_all, frac, seed=42 + seed + 10*split_id)
            train_ds = ds.make_sup_dataset(sub_tr_idx, img_size=IMG_PCAM_VIT)
            train_dl = DataLoader(train_ds, batch_size=BATCH_SUP_VIT, shuffle=True, num_workers=use_workers, pin_memory=PIN_MEMORY)

            train_feats, train_y = extract_features(backbone, train_dl)
            val_feats, val_y = extract_features(backbone, val_dl)

            mu, sd = fit_standardizer(train_feats)         # ✅ TRAIN only
            train_std = apply_standardizer(train_feats, mu, sd)
            val_std = apply_standardizer(val_feats, mu, sd) # ✅ same params

            clf = train_probe(train_std, train_y, PROBE_EPOCHS, LR_PROBE)
            metrics = eval_probe(clf, val_std, val_y)

            row = {
                "dataset": "pcam",
                "method": "simclr",
                "backbone": "vit_b_16",
                "img_size": IMG_PCAM_VIT,
                "split_id": split_id,
                "seed": seed,
                "label_frac": frac,
                "ssl_batch_used": used_bs,
                **metrics
            }
            rows.append(row)
            append_checkpoint(rows)

results_raw = pd.DataFrame(rows)
raw_path = "/kaggle/working/results_raw_vit_pcam_simclr.csv"
results_raw.to_csv(raw_path, index=False)
print("\nSaved:", raw_path)
print("Partial:", PARTIAL_RAW_PATH)
print(results_raw.head())
